# Demographics

In [ ]:

import pandas as pd
import numpy as np

## Get subjects list


In [ ]:

SUBJ_LIST = 'subjects...txt'


with open(SUBJ_LIST, 'r') as f: 
    subjects_to_include = [line.strip() for line in f.readlines()]

print(f"Number of subjects to consider: {len(subjects_to_include)}")
print("First few subjects:", subjects_to_include[:10])


Number of subjects to consider: 866
First few subjects: ['100206', '100307', '100408', '101006', '101309', '101915', '102008', '102109', '102311', '102513']


## Demographics (MZ, DZ, Singletons, Race, Age, Sex)

In [ ]:
  
RESTR_PATH = '...restricted.csv'
UNRESTR_PATH = '/...unrestricted.csv'  

df_restr = pd.read_csv(RESTR_PATH) 
df_unrestr = pd.read_csv(UNRESTR_PATH) 
df = pd.merge(df_unrestr, df_restr, on='Subject')
  
df_final = df[df['Subject'].isin(subjects_to_include)].copy()
total_n = len(df_final)

# Keep only GT Zygosity 

df_final['ZygosityGT'] = df_final['ZygosityGT'].replace('', np.nan) 
gt_available = df_final['ZygosityGT'].sum()
gt_pct = (gt_available / total_n) * 100
   
family_groups = df_final.groupby('Family_ID')

# Count MZ DZ and Singletons 

mz_pairs = 0
dz_pairs = 0
singletons = 0

for fam_id, group in family_groups: 
    group_gt = group[group['ZygosityGT']]
    
    if len(group) == 1:
        singletons += 1
        continue
 
    mzs = group_gt[group_gt['ZygosityGT'] == 'MZ']
    mz_pairs += len(mzs) // 2 

    dzs = group_gt[group_gt['ZygosityGT'] == 'DZ']
    dz_pairs += len(dzs) // 2
 
mz_subjects = mz_pairs * 2
dz_subjects = dz_pairs * 2
 
mz_pct = (mz_subjects / total_n) * 100
dz_pct = (dz_subjects / total_n) * 100
singleton_pct = (singletons / total_n) * 100
 
mz_gt_count = (df_final['ZygosityGT'] == 'MZ').sum()
dz_gt_count = (df_final['ZygosityGT'] == 'DZ').sum()

mz_gt_pct = (mz_gt_count / total_n) * 100
dz_gt_pct = (dz_gt_count / total_n) * 100

# Count Age, Race and Sex  
 
white_pct = (len(df_final[df_final['Race'] == 'White']) / total_n) * 100

age_mean = df_final['Age_in_Yrs'].mean() 
age_sd = df_final['Age_in_Yrs'].std()

female_pct = (len(df_final[df_final['Gender'] == 'F']) / total_n) * 100

 
print(f"Total N: {total_n}")
print("-" * 50)

print(f"GT Available: {gt_available} ({gt_pct:.1f}%)")

print("\n--- Zygosity (GT labels, subject-level) ---")
print(f"MZ subjects (GT): {mz_gt_count} ({mz_gt_pct:.1f}%)")
print(f"DZ subjects (GT): {dz_gt_count} ({dz_gt_pct:.1f}%)")

print("\n--- Pair-based counts ---")
print(f"Complete MZ pairs: {mz_pairs} ({mz_pct:.1f}% of subjects)")
print(f"Complete DZ pairs: {dz_pairs} ({dz_pct:.1f}% of subjects)")
print(f"Singletons: {singletons} ({singleton_pct:.1f}%)")

print("\n--- Demographics ---")
print(f"Race: White {white_pct:.1f}%")
print(f"Age: {age_mean:.2f} (SD={age_sd:.2f})")
print(f"Gender: {female_pct:.1f}% Female")


Total N: 866
--------------------------------------------------
GT Available: 866 (100.0%)

--- Zygosity (GT labels, subject-level) ---
MZ subjects (GT): 211 (24.4%)
DZ subjects (GT): 136 (15.7%)

--- Pair-based counts ---
Complete MZ pairs: 78 (18.0% of subjects)
Complete DZ pairs: 55 (12.7% of subjects)
Singletons: 116 (13.4%)

--- Demographics ---
Race: White 74.1%
Age: 28.74 (SD=3.76)
Gender: 57.6% Female
